# Day 3 · SQL 集训日

> **目标**：从"会 SELECT JOIN"升级到"能写复杂分析查询"

> **武器**：DuckDB —— 一个能对 CSV/Parquet 直接跑 SQL 的内嵌数据库

## 为什么 SQL 是数据工程师的核心武器

在你后面要接触的所有大数据系统里：

- **Hive** 用 HQL（SQL 方言）
- **Spark** 用 SparkSQL
- **Flink** 用 FlinkSQL
- **ClickHouse** 用 SQL
- **MySQL / PostgreSQL** 当然也是 SQL

**SQL 是数据世界的世界语**。你今天写的每一行 SQL，明天都能直接拿去 Hive 上跑（90% 不用改）。

## 今天的训练强度

比 Day 2 更累 —— Day 2 是手动密集型（点点点装东西），今天是脑力密集型（每道题都要想）。**每完成一节休息 10 分钟**，效率比硬撑高 3 倍。

---
# Hour 1 · DuckDB 上手与 SQL 复习

## 0.1 为什么用 DuckDB

传统的学 SQL 流程：装 MySQL → 配权限 → 导入数据 → 跑查询。整个链路下来 1 小时还没开始写 SQL。

**DuckDB 的革命**：`import duckdb` 之后**直接对你的 CSV 文件跑 SQL**，0 配置。

它的语法跟 PostgreSQL / Hive / Spark SQL 高度一致 —— 今天你写的 SQL，Day 4 进 Hive 几乎可以原样复用。

In [7]:
import duckdb
import pandas as pd

# DuckDB 的连接对象（内存版,够今天用）
con = duckdb.connect()

# pandas 显示配置
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 200)

print(f'DuckDB version: {duckdb.__version__}')
print('准备就绪')

DuckDB version: 1.3.2
准备就绪


## 0.2 把数据注册成表

DuckDB 可以直接 `read_csv('文件路径')` 来查 CSV，但每次都写路径太啰嗦。我们把这些 CSV **注册成 SQL 表**，后面直接 `FROM movies` 即可。

In [8]:
# 注册 IMDb 数据为表（DuckDB 直接读 .tsv.gz 不用解压!）
con.execute("""
CREATE OR REPLACE VIEW imdb_titles AS 
SELECT * FROM read_csv('../data/raw/imdb/title.basics.tsv.gz',
    delim='\t',
    nullstr='\\N',
    quote=''
)
""")

con.execute("""
CREATE OR REPLACE VIEW imdb_ratings AS
SELECT * FROM read_csv('../data/raw/imdb/title.ratings.tsv.gz',
    delim='\t',
    nullstr='\\N'
)
""")

# 注册 MovieLens
con.execute("CREATE OR REPLACE VIEW ml_movies AS SELECT * FROM read_csv('../data/raw/movielens/ml-25m/movies.csv')")
con.execute("CREATE OR REPLACE VIEW ml_ratings AS SELECT * FROM read_csv('../data/raw/movielens/ml-25m/ratings.csv')")
con.execute("CREATE OR REPLACE VIEW ml_tags AS SELECT * FROM read_csv('../data/raw/movielens/ml-25m/tags.csv')")
con.execute("CREATE OR REPLACE VIEW ml_links AS SELECT * FROM read_csv('../data/raw/movielens/ml-25m/links.csv')")

print('✅ 注册了 6 个表/视图')
con.sql("SHOW TABLES").show()

✅ 注册了 6 个表/视图
┌──────────────┐
│     name     │
│   varchar    │
├──────────────┤
│ imdb_ratings │
│ imdb_titles  │
│ ml_links     │
│ ml_movies    │
│ ml_ratings   │
│ ml_tags      │
└──────────────┘



## 0.3 第一个 SQL 查询

**DuckDB 调用模式**：
- `con.sql("...")` 返回 DuckDB 的结果对象,后面接 `.show()` 打印或 `.df()` 转 pandas
- 在 Notebook 里推荐 `.df()` 转 pandas,因为 pandas 表格更好看且能用 head/tail

In [9]:
# 看一下每张表的形态
con.sql("SELECT * FROM imdb_titles LIMIT 3").df()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,None,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,None,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,None,5,"Animation,Comedy,Romance"


In [4]:
con.sql("SELECT * FROM ml_ratings LIMIT 3").df()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828


## 0.4 SQL 复习：六大子句的执行顺序

**这是 SQL 最重要的基础知识,99% 的人都搞反过**：

你**写**的顺序：
```sql
SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ...
```

数据库**执行**的顺序：
```
1. FROM       —— 先找到表
2. WHERE      —— 过滤行（这步还没分组）
3. GROUP BY   —— 分组
4. HAVING     —— 过滤分组（聚合之后再过滤）
5. SELECT     —— 计算列
6. ORDER BY   —— 排序
7. LIMIT      —— 截取
```

**记住这个顺序**,后面所有进阶 SQL 都靠它。这就是为什么 WHERE 里不能用 SELECT 计算出的列别名（WHERE 比 SELECT 先执行）。

## 0.5 复习题 Q1-Q3

下面 3 道题先**自己写**,写完看下一格答案对照。

### Q1：从 imdb_ratings 表中,找出投票数 >= 100,000 的所有作品,按评分降序排列前 20

提示用到的字段：`tconst`、`averageRating`、`numVotes`

In [ ]:
# 自己写：
con.sql("""
-- 你的 SQL 写在这里
    select tconst, averageRating, numVotes 
    from imdb_ratings 
    where numVotes >= 100000 
    order by averageRating desc 
    limit 20
""").df()

,tconst,averageRating,numVotes
0,tt10888594,1.8,181732
1,tt6208148,2.2,396349
2,tt0799949,2.5,113861
3,tt12915716,2.6,137593
4,tt1073498,2.8,113248
5,tt10886166,3.3,108934
6,tt0327554,3.5,132517
7,tt0368226,3.6,100066
8,tt0118688,3.8,283012
9,tt0938283,3.9,181447


In [10]:
# Q1 参考答案
con.sql("""
SELECT tconst, averageRating, numVotes
FROM imdb_ratings
WHERE numVotes >= 100000
ORDER BY averageRating DESC
LIMIT 20
""").df()

,tconst,averageRating,numVotes
0,tt2178784,9.9,182168
1,tt4283094,9.9,227600
2,tt4283088,9.9,308601
3,tt13605714,9.8,103691
4,tt13819632,9.8,103934
5,tt3866850,9.8,142559
6,tt17594170,9.8,102610
7,tt9898836,9.8,120065
8,tt2301455,9.8,291778
9,tt9906260,9.8,166514


### Q2：把 imdb_titles 和 imdb_ratings 关联起来,找出 1990 年之后的高分电影（averageRating >= 8.5, numVotes >= 100000）

需要 JOIN + WHERE 多条件。

In [11]:
# 自己写：
con.sql("""
    select t.tconst,t.startYear
    from imdb_titles t
    inner join imdb_ratings r on t.tconst=r.tconst
    where startYear>1990
    and averageRating>=8.5
    AND numVotes >= 100000
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,tconst,startYear
0,tt0436992,2005
1,tt0458290,2008
2,tt0468569,2008
3,tt0472954,2005
4,tt0482571,2006
...,...,...
176,tt6763664,2018
177,tt8420184,2020
178,tt9544034,2019
179,tt9898836,2019


In [12]:
# Q2 参考答案
con.sql("""
SELECT 
    t.primaryTitle,
    t.startYear,
    t.genres,
    r.averageRating,
    r.numVotes
FROM imdb_titles t
INNER JOIN imdb_ratings r ON t.tconst = r.tconst
WHERE t.titleType = 'movie'
  AND t.startYear >= 1990
  AND r.averageRating >= 8.5
  AND r.numVotes >= 100000
ORDER BY r.averageRating DESC, r.numVotes DESC
LIMIT 30
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,primaryTitle,startYear,genres,averageRating,numVotes
0,The Shawshank Redemption,1994,Drama,9.3,3197412
1,The Dark Knight,2008,"Crime,Thriller",9.1,3178266
2,The Lord of the Rings: The Return of the King,2003,"Adventure,Drama,Fantasy",9.0,2169411
3,Schindler's List,1993,"Biography,Drama,History",9.0,1587775
4,The Lord of the Rings: The Fellowship of the Ring,2001,"Adventure,Drama,Fantasy",8.9,2211568
5,Inception,2010,"Adventure,Sci-Fi,Thriller",8.8,2825901
6,Fight Club,1999,"Crime,Drama,Thriller",8.8,2617971
7,Forrest Gump,1994,"Drama,Romance",8.8,2503025
8,Pulp Fiction,1994,"Crime,Drama",8.8,2440694
9,The Lord of the Rings: The Two Towers,2002,"Adventure,Drama,Fantasy",8.8,1961687


### Q3：统计 ml_ratings 表里每个用户打了多少分,以及他们的平均分,只显示打分超过 100 次的用户前 20 名

用到 GROUP BY + HAVING + ORDER BY + LIMIT。

**思考**：HAVING 和 WHERE 的区别是什么？

In [15]:
# 自己写：
con.sql("""
select 
    userID,
    count(*) as rating_count,
    round(avg(rating),2) as avg_rating
from ml_ratings
group by userID
having count(*) >100
order by rating_count desc
limit 20        
""").df()

,userId,rating_count,avg_rating
0,72315,32202,3.08
1,80974,9178,3.28
2,137293,8913,3.18
3,33844,7919,2.58
4,20055,7488,3.21
5,109731,6647,2.82
6,92046,6564,3.48
7,49403,6553,1.52
8,30879,5693,2.88
9,115102,5649,2.46


In [14]:
# Q3 参考答案
con.sql("""
SELECT 
    userId,
    COUNT(*) AS rating_count,
    ROUND(AVG(rating), 2) AS avg_rating
FROM ml_ratings
GROUP BY userId
HAVING COUNT(*) > 100
ORDER BY rating_count DESC
LIMIT 20
""").df()

# 答案：WHERE 在分组前过滤行，HAVING 在分组后过滤组。
# 这里我们要按 "打分次数>100" 过滤，这是聚合后的结果，必须用 HAVING。

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,userId,rating_count,avg_rating
0,72315,32202,3.08
1,80974,9178,3.28
2,137293,8913,3.18
3,33844,7919,2.58
4,20055,7488,3.21
5,109731,6647,2.82
6,92046,6564,3.48
7,49403,6553,1.52
8,30879,5693,2.88
9,115102,5649,2.46


---
# Hour 2 · 聚合的进阶

GROUP BY 不只是 COUNT —— 还可以做**多维聚合、条件聚合、字符串聚合**。这一节让你的聚合武器库扩充 5 倍。

## 2.1 多维 GROUP BY（按多个维度同时聚合）

In [16]:
# 按年份 + 类型,统计 IMDb 上每种类型每年的产量
con.sql("""
SELECT 
    startYear,
    titleType,
    COUNT(*) AS title_count
FROM imdb_titles
WHERE startYear BETWEEN 2010 AND 2020
  AND titleType IN ('movie', 'tvSeries')
GROUP BY startYear, titleType
ORDER BY startYear, titleType
LIMIT 25
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,startYear,titleType,title_count
0,2010,movie,13068
1,2010,tvSeries,7244
2,2011,movie,14026
3,2011,tvSeries,8739
4,2012,movie,15081
5,2012,tvSeries,9537
6,2013,movie,15778
7,2013,tvSeries,9960
8,2014,movie,16934
9,2014,tvSeries,10379


## 2.2 条件聚合（FILTER 或 CASE WHEN）—— 重磅技巧

**这个模式是数据岗 SQL 题的常客**：在一次查询里同时算"满足条件 A 的数量"和"满足条件 B 的数量"。

比如：每个用户打了多少 5 星、多少 1 星？

**写法 1：CASE WHEN（通用,所有 SQL 引擎都支持）**

**写法 2：FILTER (WHERE ...)（DuckDB / PostgreSQL 支持,更优雅）**

In [17]:
# 写法 1：CASE WHEN（推荐先掌握这个,因为 Hive 也支持）
con.sql("""
SELECT 
    userId,
    COUNT(*) AS total_ratings,
    SUM(CASE WHEN rating = 5.0 THEN 1 ELSE 0 END) AS five_star_count,
    SUM(CASE WHEN rating <= 2.0 THEN 1 ELSE 0 END) AS low_rating_count,
    SUM(CASE WHEN rating >= 4.0 THEN 1 ELSE 0 END) AS positive_count
FROM ml_ratings
GROUP BY userId
HAVING COUNT(*) >= 50
ORDER BY total_ratings DESC
LIMIT 15
""").df()

,userId,total_ratings,five_star_count,low_rating_count,positive_count
0,72315,32202,738.0,3482.0,4733.0
1,80974,9178,27.0,475.0,2374.0
2,137293,8913,12.0,442.0,883.0
3,33844,7919,55.0,2909.0,872.0
4,20055,7488,435.0,1346.0,2495.0
5,109731,6647,54.0,1789.0,1156.0
6,92046,6564,874.0,861.0,2910.0
7,49403,6553,182.0,5041.0,606.0
8,30879,5693,20.0,937.0,493.0
9,115102,5649,472.0,2438.0,1231.0


In [ ]:
# 写法 2：FILTER（更简洁,但 Hive 不支持,要注意场景）
con.sql("""
SELECT 
    userId,
    COUNT(*) AS total_ratings,
    COUNT(*) FILTER (WHERE rating = 5.0) AS five_star_count,
    COUNT(*) FILTER (WHERE rating <= 2.0) AS low_rating_count,
    COUNT(*) FILTER (WHERE rating >= 4.0) AS positive_count
FROM ml_ratings
GROUP BY userId
HAVING COUNT(*) >= 50
ORDER BY total_ratings DESC
LIMIT 15
""").df()

**重要建议**：你看到了同一个分析两种写法。**优先掌握 CASE WHEN**,因为它是 SQL 标准,Hive/Spark/MySQL 都支持。FILTER 知道有这个东西就行,工作中再去用。

## 2.3 字符串聚合 STRING_AGG（合并多行成一行）

**典型场景**：每个用户最喜欢的几个电影列表

In [18]:
# 每个用户打 5 星的电影列表（取每个用户前 3 个）
con.sql("""
SELECT 
    r.userId,
    STRING_AGG(m.title, ' | ') AS five_star_movies
FROM ml_ratings r
JOIN ml_movies m ON r.movieId = m.movieId
WHERE r.rating = 5.0
  AND r.userId BETWEEN 1 AND 10
GROUP BY r.userId
ORDER BY r.userId
""").df()

,userId,five_star_movies
0,1,Pulp Fiction (1994) | Three Colors: Blue (Troi...
1,2,Braveheart (1995) | Star Wars: Episode IV - A ...
2,3,"Usual Suspects, The (1995) | Before the Rain (..."
3,4,Spirited Away (Sen to Chihiro no kamikakushi) ...
4,5,Twelve Monkeys (a.k.a. 12 Monkeys) (1995) | De...
5,6,Star Wars: Episode IV - A New Hope (1977) | Sh...
6,7,"Postman, The (Postino, Il) (1994) | Three Colo..."
7,8,Seven (a.k.a. Se7en) (1995) | Friday (1995) | ...
8,9,Jumanji (1995) | GoldenEye (1995) | Anne Frank...
9,10,"Usual Suspects, The (1995) | Star Wars: Episod..."


## 2.4 挑战题 Q4-Q5

### Q4：找出 IMDb 上每个类型（genres）的电影数量、平均评分、最高评分、最低评分

**陷阱**：`genres` 是逗号分隔字符串（如 `Action,Drama,Sci-Fi`）,你需要用 `UNNEST(STRING_SPLIT(genres, ','))` 拆开。

**约束**：只考虑电影（titleType='movie'）,且有评分（INNER JOIN ratings）,投票数 >= 1000

In [ ]:
# 自己写：
con.sql("""
    select
        UNNEST(STRING_SPLIT(t.genres, ',')) AS genre,
        count(*) as movie_num,
        round(avg(rating,2)) as avg_rating,
        max(rating)
""").df()

In [20]:
# Q4 参考答案
con.sql("""
WITH exploded AS (
    -- 第一步：把 genres 炸成多行，每行一个 genre
    SELECT 
        t.tconst,
        r.averageRating,
        r.numVotes,
        UNNEST(STRING_SPLIT(t.genres, ',')) AS genre
    FROM imdb_titles t
    JOIN imdb_ratings r ON t.tconst = r.tconst
    WHERE t.titleType = 'movie'
      AND r.numVotes >= 1000
      AND t.genres IS NOT NULL
)
-- 第二步：在炸开后的结果上聚合
SELECT 
    genre,
    COUNT(*) AS movie_count,
    ROUND(AVG(averageRating), 2) AS avg_rating,
    MAX(averageRating) AS max_rating,
    MIN(averageRating) AS min_rating
FROM exploded
GROUP BY genre
ORDER BY movie_count DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,genre,movie_count,avg_rating,max_rating,min_rating
0,Drama,26998,6.49,10.0,1.0
1,Comedy,16120,6.15,10.0,1.0
2,Action,8816,5.92,9.7,1.2
3,Romance,8064,6.39,9.5,1.1
4,Crime,8007,6.30,9.4,1.2
5,Thriller,7394,5.82,9.4,1.2
6,Horror,6564,5.24,9.1,1.3
7,Adventure,5263,6.10,9.1,1.0
8,Mystery,4104,5.97,9.4,1.4
9,Fantasy,2610,5.93,9.0,1.2


### Q5：评分行为画像 —— 在 ml_ratings 表上,计算每个用户的：
- 总评分次数
- 给 5 星的次数
- 给 1 星或以下的次数（**严苛用户**指标）
- 给好评的比例（评分 >= 4 的占比）
- 把用户分类成 'severe'（严苛 > 10%）/ 'lenient'（好评 > 80%）/ 'normal'（其他）

**只看打分次数 >= 50 的用户,前 20 名**

提示：分类那一列用 CASE WHEN 嵌套写。

In [ ]:
# 自己写：
con.sql("""

""").df()

In [21]:
# Q5 参考答案
con.sql("""
SELECT 
    userId,
    COUNT(*) AS total,
    SUM(CASE WHEN rating = 5.0 THEN 1 ELSE 0 END) AS five_star,
    SUM(CASE WHEN rating <= 1.0 THEN 1 ELSE 0 END) AS low_rating,
    ROUND(100.0 * SUM(CASE WHEN rating >= 4.0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS positive_pct,
    ROUND(100.0 * SUM(CASE WHEN rating <= 1.0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS severe_pct,
    CASE 
        WHEN 100.0 * SUM(CASE WHEN rating <= 1.0 THEN 1 ELSE 0 END) / COUNT(*) > 10 THEN 'severe'
        WHEN 100.0 * SUM(CASE WHEN rating >= 4.0 THEN 1 ELSE 0 END) / COUNT(*) > 80 THEN 'lenient'
        ELSE 'normal'
    END AS user_type
FROM ml_ratings
GROUP BY userId
HAVING COUNT(*) >= 50
ORDER BY total DESC
LIMIT 20
""").df()

,userId,total,five_star,low_rating,positive_pct,severe_pct,user_type
0,72315,32202,738.0,509.0,14.7,1.6,normal
1,80974,9178,27.0,3.0,25.9,0.0,normal
2,137293,8913,12.0,34.0,9.9,0.4,normal
3,33844,7919,55.0,1226.0,11.0,15.5,severe
4,20055,7488,435.0,72.0,33.3,1.0,normal
5,109731,6647,54.0,450.0,17.4,6.8,normal
6,92046,6564,874.0,222.0,44.3,3.4,normal
7,49403,6553,182.0,4198.0,9.2,64.1,severe
8,30879,5693,20.0,79.0,8.7,1.4,normal
9,115102,5649,472.0,1436.0,21.8,25.4,severe


---
# Hour 3 · 子查询 + CTE

## 3.1 为什么需要它们

当一个分析需要 **"先算出中间结果,再基于中间结果查询"** 时,你需要子查询或 CTE。

**例如**：找出"评分高于其类型平均分的电影"——
1. 先算每个类型的平均分（中间结果）
2. 再用每部电影的评分去跟它所在类型的平均分对比

## 3.2 子查询 vs CTE 的选择

- **子查询**：简单嵌套,写一次用一次,适合一两层简单嵌套
- **CTE（WITH ... AS）**：可读性强,中间结果有名字,适合复杂多层分析

**数据工程师 90% 的复杂查询都用 CTE**——因为生产环境的 SQL 经常 100 行以上,没有 CTE 写不动。

## 3.3 简单子查询

In [ ]:
# 找出评分高于全平台平均分的电影
con.sql("""
SELECT 
    t.primaryTitle, 
    t.startYear, 
    r.averageRating
FROM imdb_titles t
JOIN imdb_ratings r ON t.tconst = r.tconst
WHERE t.titleType = 'movie'
  AND r.numVotes >= 50000
  AND r.averageRating > (
      -- 子查询：全平台平均分
      SELECT AVG(averageRating) FROM imdb_ratings WHERE numVotes >= 50000
  )
ORDER BY r.averageRating DESC
LIMIT 15
""").df()

## 3.4 CTE：让查询变成"故事"

**CTE 的本质：给中间结果起一个名字,让 SQL 像写函数一样可读。**

**关键句式**：
```sql
WITH step1 AS (
    -- 第一步查询
),
step2 AS (
    -- 第二步,可以用 step1
    SELECT ... FROM step1
)
SELECT ... FROM step2;
```

下面用 CTE 实现同样的需求,但加一个增强：**找出评分高于其类型平均分的电影**

In [ ]:
con.sql("""
WITH movie_with_genre AS (
    -- 第一步：把电影展开成（电影,类型）一对多关系
    SELECT 
        t.tconst,
        t.primaryTitle,
        t.startYear,
        r.averageRating,
        r.numVotes,
        UNNEST(STRING_SPLIT(t.genres, ',')) AS genre
    FROM imdb_titles t
    JOIN imdb_ratings r ON t.tconst = r.tconst
    WHERE t.titleType = 'movie'
      AND r.numVotes >= 10000
      AND t.genres IS NOT NULL
),
genre_avg AS (
    -- 第二步：算每个类型的平均分
    SELECT 
        genre,
        AVG(averageRating) AS genre_avg_rating
    FROM movie_with_genre
    GROUP BY genre
)
-- 主查询：跟类型平均分对比
SELECT 
    m.primaryTitle,
    m.startYear,
    m.genre,
    m.averageRating,
    ROUND(g.genre_avg_rating, 2) AS genre_avg,
    ROUND(m.averageRating - g.genre_avg_rating, 2) AS above_avg
FROM movie_with_genre m
JOIN genre_avg g ON m.genre = g.genre
WHERE m.averageRating > g.genre_avg_rating + 2.0  -- 超出类型均分 2 分以上
ORDER BY above_avg DESC
LIMIT 20
""").df()

**感受到 CTE 的威力了吗？** 这一段 SQL 读起来像三段独立的话：
1. "先把电影展开成多类型行"
2. "再算每个类型的平均分"
3. "最后找出比类型均分高的电影"

如果用子查询嵌套写,会变成俄罗斯套娃,谁也读不动。

## 3.5 挑战题 Q6-Q7

### Q6（CTE 练习）：找出"评分超过用户自己历史平均分的电影"

**步骤拆解**（自己写 CTE）：
1. CTE 1：算每个用户的历史平均分
2. 主查询：JOIN 上 ml_ratings,过滤评分 > 用户均分 + 1 的记录
3. 关联 ml_movies 拿到电影名,显示前 20

In [ ]:
# 自己写：
con.sql("""
WITH ___ AS (
    ___
)
SELECT ___
""").df()

In [ ]:
# Q6 参考答案
con.sql("""
WITH user_avg AS (
    SELECT 
        userId,
        AVG(rating) AS personal_avg
    FROM ml_ratings
    GROUP BY userId
)
SELECT 
    r.userId,
    m.title,
    r.rating,
    ROUND(u.personal_avg, 2) AS personal_avg,
    ROUND(r.rating - u.personal_avg, 2) AS exceeds_by
FROM ml_ratings r
JOIN user_avg u ON r.userId = u.userId
JOIN ml_movies m ON r.movieId = m.movieId
WHERE r.rating > u.personal_avg + 1.5
  AND r.userId BETWEEN 1 AND 100  -- 限定用户范围加快查询
ORDER BY exceeds_by DESC, r.userId
LIMIT 20
""").df()

### Q7（多层 CTE）：找出"打过 5 星给冷门电影的用户"

**业务含义**：这些用户是小众品味爱好者,做内容平台时是"长尾内容"的关键用户。

**步骤**：
1. CTE 1：算每个电影的总评分次数（衡量是否冷门）
2. CTE 2：定义冷门电影 = 评分次数 < 50 次
3. 主查询：找出给这些冷门电影打过 5 星的用户,统计他们打了多少个 5 星冷门片

In [ ]:
# 自己写：
con.sql("""
WITH ___ AS (
    ___
),
___ AS (
    ___
)
SELECT ___
""").df()

In [ ]:
# Q7 参考答案
con.sql("""
WITH movie_popularity AS (
    SELECT 
        movieId,
        COUNT(*) AS rating_count
    FROM ml_ratings
    GROUP BY movieId
),
obscure_movies AS (
    SELECT movieId
    FROM movie_popularity
    WHERE rating_count < 50
)
SELECT 
    r.userId,
    COUNT(*) AS obscure_five_star_count
FROM ml_ratings r
JOIN obscure_movies o ON r.movieId = o.movieId
WHERE r.rating = 5.0
GROUP BY r.userId
HAVING COUNT(*) >= 5  -- 至少给 5 部冷门片打过 5 星
ORDER BY obscure_five_star_count DESC
LIMIT 20
""").df()

---
# Hour 4 · ⭐ 窗口函数 ⭐

## 4.1 窗口函数：数据岗的灵魂武器

**面试时如果你说"我熟练 SQL"但不会窗口函数,面试官会当场否定你。** 窗口函数是数据分析 SQL 的分水岭。

## 4.2 窗口函数解决什么问题

GROUP BY 会**"压扁"行**——10 个用户的 100 行明细,GROUP BY userId 后变成 10 行汇总,明细就没了。

**窗口函数的革命**：**保留每一行明细的同时,计算汇总值**。

**典型场景**：
- 每个用户的每条评分,加一列"这是他第几次打分"——`ROW_NUMBER()`
- 每条评分加一列"他之前一次打分时间"——`LAG()`
- 每条评分加一列"他到目前为止的总打分次数"——`COUNT() OVER`

## 4.3 通用语法

```sql
窗口函数() OVER (
    PARTITION BY 分组列      -- 类似 GROUP BY,但不压扁
    ORDER BY 排序列         -- 在每个分组内的排序
    ROWS BETWEEN ... AND ... -- 窗口范围（可选）
)
```

## 4.4 ROW_NUMBER / RANK / DENSE_RANK：三兄弟

三个都是"排名",但行为不同：

| 函数 | 并列处理 | 例：100,90,90,80 → 排名 |
|---|---|---|
| ROW_NUMBER | 不允许并列,强行分先后 | 1, 2, 3, 4 |
| RANK | 允许并列,跳号 | 1, 2, 2, 4 |
| DENSE_RANK | 允许并列,不跳号 | 1, 2, 2, 3 |

In [22]:
# 经典场景：每个类型评分前 3 的电影（TopN per group）
con.sql("""
WITH movie_genre AS (
    SELECT 
        t.primaryTitle,
        t.startYear,
        r.averageRating,
        r.numVotes,
        UNNEST(STRING_SPLIT(t.genres, ',')) AS genre
    FROM imdb_titles t
    JOIN imdb_ratings r ON t.tconst = r.tconst
    WHERE t.titleType = 'movie' 
      AND r.numVotes >= 100000
      AND t.genres IS NOT NULL
),
ranked AS (
    SELECT 
        primaryTitle, 
        startYear, 
        genre, 
        averageRating,
        ROW_NUMBER() OVER (PARTITION BY genre ORDER BY averageRating DESC, numVotes DESC) AS rn
    FROM movie_genre
)
SELECT * FROM ranked WHERE rn <= 3
ORDER BY genre, rn
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,primaryTitle,startYear,genre,averageRating,rn
0,The Matrix,1999,Action,8.7,1
1,Terminator 2: Judgment Day,1991,Action,8.6,2
2,Seven Samurai,1954,Action,8.6,3
3,The Lord of the Rings: The Return of the King,2003,Adventure,9.0,1
4,The Lord of the Rings: The Fellowship of the Ring,2001,Adventure,8.9,2
...,...,...,...,...,...
61,Casablanca,1942,War,8.5,2
62,Grave of the Fireflies,1988,War,8.5,3
63,"The Good, the Bad and the Ugly",1966,Western,8.8,1
64,Django Unchained,2012,Western,8.5,2


**这就是"TopN per group"**——数据岗高频面试题。**用 ROW_NUMBER + PARTITION BY 三行解决**。

记住这个模式,Day 5 数仓建模和后面所有分析都会用到。

## 4.5 LAG / LEAD：前一行/后一行

**LAG(列,n)**：取分组内当前行**往前 n 行**的值

**LEAD(列,n)**：取分组内当前行**往后 n 行**的值

**典型场景**：算"两次打分的时间间隔"

In [ ]:
con.sql("""
SELECT 
    userId,
    movieId,
    rating,
    to_timestamp(timestamp) AS rating_time,
    to_timestamp(LAG(timestamp) OVER (PARTITION BY userId ORDER BY timestamp)) AS prev_rating_time,
    timestamp - LAG(timestamp) OVER (PARTITION BY userId ORDER BY timestamp) AS gap_seconds
FROM ml_ratings
WHERE userId = 1  -- 看用户 1 的连续打分时序
ORDER BY timestamp
LIMIT 15
""").df()

**观察**：
- 第一行的 `prev_rating_time` 是 NaT/None（没有前一行）
- `gap_seconds` 是两次打分的间隔秒数
- 这是分析用户活跃模式的核心 SQL,会用在 "session 切分" 和 "用户活跃周期" 分析

## 4.6 累计聚合（SUM/COUNT/AVG OVER ORDER BY）

**典型场景**：每天的累计打分量,而不是当天打分量

In [ ]:
con.sql("""
WITH daily AS (
    SELECT 
        date_trunc('day', to_timestamp(timestamp)) AS day,
        COUNT(*) AS daily_count
    FROM ml_ratings
    WHERE timestamp >= 1000000000  -- 2001-09 之后
    GROUP BY day
)
SELECT 
    day,
    daily_count,
    SUM(daily_count) OVER (ORDER BY day) AS cumulative_total,
    AVG(daily_count) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS moving_avg_7day
FROM daily
ORDER BY day
LIMIT 20
""").df()

**这里的两个窗口函数极其重要,工作中天天用**：

1. **累计求和**：`SUM() OVER (ORDER BY day)`——一直加,做趋势图必用
2. **滑动窗口**：`AVG() OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)`——"最近 7 天"的移动平均,用来**平滑掉周期性波动**

**面试时被问"怎么算最近 N 天的移动平均",这就是标准答案。**

## 4.7 挑战题 Q8-Q10

### Q8：每个用户的第一次和最后一次打分,以及打分时长跨度

**步骤**：
1. 用 ROW_NUMBER 排序每个用户的打分,正序+倒序
2. 过滤出 rn=1（第一条）
3. 用 MAX/MIN 算时长

**或者更简洁**：直接 GROUP BY userId,用 MIN(timestamp) 和 MAX(timestamp)。**但请你两种都写一遍**,理解窗口函数的等价表达。

In [ ]:
# 自己写（用窗口函数版本）：
con.sql("""

""").df()

In [ ]:
# Q8 参考答案：用 GROUP BY 写最简洁
con.sql("""
SELECT 
    userId,
    COUNT(*) AS rating_count,
    to_timestamp(MIN(timestamp)) AS first_rating,
    to_timestamp(MAX(timestamp)) AS last_rating,
    (MAX(timestamp) - MIN(timestamp)) / 86400.0 AS active_days
FROM ml_ratings
GROUP BY userId
HAVING COUNT(*) >= 100
ORDER BY active_days DESC
LIMIT 15
""").df()

### Q9：找每年评分最高的电影（每年 Top 1）

**典型 TopN per group 题**——用 ROW_NUMBER。

约束：只看电影(`titleType='movie'`),投票数 >= 50,000,1990 年之后。

In [ ]:
# 自己写：
con.sql("""

""").df()

In [ ]:
# Q9 参考答案
con.sql("""
WITH ranked AS (
    SELECT 
        t.startYear,
        t.primaryTitle,
        r.averageRating,
        r.numVotes,
        ROW_NUMBER() OVER (PARTITION BY t.startYear ORDER BY r.averageRating DESC, r.numVotes DESC) AS rn
    FROM imdb_titles t
    JOIN imdb_ratings r ON t.tconst = r.tconst
    WHERE t.titleType = 'movie' 
      AND r.numVotes >= 50000
      AND t.startYear >= 1990
      AND t.startYear IS NOT NULL
)
SELECT startYear, primaryTitle, averageRating, numVotes
FROM ranked
WHERE rn = 1
ORDER BY startYear DESC
LIMIT 20
""").df()

### Q10（综合题）：每个用户的"打分时间间隔"分析

**业务问题**：用户的打分活动有什么节奏？是密集打一阵子还是均匀分布？

**步骤**：
1. 用 LAG 算每条记录跟上一条的时间间隔
2. GROUP BY userId 算这个间隔的中位数、最大值、最小值
3. 找出"短时间内大量打分"的用户（中位间隔很小）

提示：DuckDB 里中位数函数是 `MEDIAN(...)` 或 `PERCENTILE_CONT(0.5) WITHIN GROUP (...)`

In [ ]:
# 自己写：
con.sql("""

""").df()

In [ ]:
# Q10 参考答案
con.sql("""
WITH gaps AS (
    SELECT 
        userId,
        timestamp - LAG(timestamp) OVER (PARTITION BY userId ORDER BY timestamp) AS gap_sec
    FROM ml_ratings
)
SELECT 
    userId,
    COUNT(*) AS rating_count,
    MEDIAN(gap_sec) / 60.0 AS median_gap_minutes,
    MIN(gap_sec) AS min_gap_sec,
    MAX(gap_sec) / 86400.0 AS max_gap_days
FROM gaps
WHERE gap_sec IS NOT NULL
GROUP BY userId
HAVING COUNT(*) >= 100
ORDER BY median_gap_minutes ASC
LIMIT 15
""").df()

**休息一下 ☕**

你已经完成了 SQL 学习中最难的一关。窗口函数搞清楚之后,后面所有 SQL 题对你来说都是组合现有武器。

---
# Hour 5 · CASE WHEN 与场景化分析

## 5.1 CASE WHEN 在数据岗的角色

CASE WHEN 看起来简单（就是 if-else）,但是数据岗 80% 的"指标定义"都靠它：
- 把连续值"分桶"成分类（年龄段、评分段）
- 把多种状态合并成几个标签
- 实现行级条件聚合（前面已经用过）

## 5.2 经典案例：电影年代分桶

In [ ]:
con.sql("""
WITH bucketed AS (
    SELECT 
        primaryTitle,
        startYear,
        CASE 
            WHEN startYear < 1950 THEN '黄金时代(<1950)'
            WHEN startYear < 1980 THEN '经典时代(1950-1980)'
            WHEN startYear < 2000 THEN '现代时代(1980-2000)'
            WHEN startYear < 2010 THEN '新世纪初(2000-2010)'
            ELSE '当代(2010+)'
        END AS era
    FROM imdb_titles
    WHERE titleType = 'movie' AND startYear IS NOT NULL
)
SELECT 
    era,
    COUNT(*) AS movie_count
FROM bucketed
GROUP BY era
ORDER BY MIN(CASE 
    WHEN era = '黄金时代(<1950)' THEN 1
    WHEN era = '经典时代(1950-1980)' THEN 2
    WHEN era = '现代时代(1980-2000)' THEN 3
    WHEN era = '新世纪初(2000-2010)' THEN 4
    ELSE 5
END)
""").df()

## 5.3 综合题 Q11-Q12

### Q11：用户活跃度分层

**业务问题**：根据用户打分次数,把用户分成 "超活跃 / 活跃 / 普通 / 流失" 四档,统计每档有多少人,他们的平均评分如何。

**分层规则**：
- 超活跃：>= 1000 次
- 活跃：100-999 次
- 普通：20-99 次
- 流失：< 20 次

In [ ]:
# 自己写：
con.sql("""

""").df()

In [ ]:
# Q11 参考答案
con.sql("""
WITH user_stats AS (
    SELECT 
        userId,
        COUNT(*) AS rating_count,
        AVG(rating) AS avg_rating
    FROM ml_ratings
    GROUP BY userId
),
tiered AS (
    SELECT 
        userId,
        rating_count,
        avg_rating,
        CASE 
            WHEN rating_count >= 1000 THEN '1_超活跃'
            WHEN rating_count >= 100 THEN '2_活跃'
            WHEN rating_count >= 20 THEN '3_普通'
            ELSE '4_流失'
        END AS tier
    FROM user_stats
)
SELECT 
    tier,
    COUNT(*) AS user_count,
    SUM(rating_count) AS total_ratings_contributed,
    ROUND(AVG(rating_count), 1) AS avg_ratings_per_user,
    ROUND(AVG(avg_rating), 2) AS avg_score_given
FROM tiered
GROUP BY tier
ORDER BY tier
""").df()

### Q12：电影口碑分类

把每部 IMDb 电影按**评分 × 投票数**双维度分类：

| 评分 | 投票数 | 分类 |
|---|---|---|
| >=8.0 | >=100k | 神作 |
| >=7.0 | >=50k | 佳作 |
| >=6.0 | >=10k | 良作 |
| <6.0 | >=10k | 平庸 |
| >=7.0 | <10k | 小众佳作 |
| 其他 | | 待考察 |

统计每个分类的数量和年代分布。

In [ ]:
# Q12 自己写
con.sql("""

""").df()

In [ ]:
# Q12 参考答案
con.sql("""
WITH classified AS (
    SELECT 
        t.startYear,
        r.averageRating,
        r.numVotes,
        CASE 
            WHEN r.averageRating >= 8.0 AND r.numVotes >= 100000 THEN '神作'
            WHEN r.averageRating >= 7.0 AND r.numVotes >= 50000 THEN '佳作'
            WHEN r.averageRating >= 6.0 AND r.numVotes >= 10000 THEN '良作'
            WHEN r.averageRating < 6.0 AND r.numVotes >= 10000 THEN '平庸'
            WHEN r.averageRating >= 7.0 AND r.numVotes < 10000 THEN '小众佳作'
            ELSE '待考察'
        END AS quality_tier
    FROM imdb_titles t
    JOIN imdb_ratings r ON t.tconst = r.tconst
    WHERE t.titleType = 'movie' AND t.startYear IS NOT NULL
)
SELECT 
    quality_tier,
    COUNT(*) AS movie_count,
    ROUND(AVG(averageRating), 2) AS avg_rating,
    ROUND(AVG(numVotes)) AS avg_votes
FROM classified
GROUP BY quality_tier
ORDER BY movie_count DESC
""").df()

---
# Hour 6 · ⭐ 数据工程经典题 ⭐

这一节的题目**完全对应你后面面试和工作中要写的 SQL**。每一题都是数据岗高频面试题原型。

## 6.1 留存率分析

**业务问题**：2010 年首次打分的用户,在 2011/2012/2013 年还继续活跃的有多少？这就是**用户留存**。

**算法逻辑**：
1. CTE 1：找出每个用户的首次打分年份
2. CTE 2：找出每个用户打过分的所有年份
3. 主查询：以"首年"分组,统计后续每年还在的用户数

In [ ]:
con.sql("""
WITH user_first_year AS (
    SELECT 
        userId,
        MIN(EXTRACT(YEAR FROM to_timestamp(timestamp))) AS first_year
    FROM ml_ratings
    GROUP BY userId
),
user_active_years AS (
    SELECT DISTINCT
        userId,
        EXTRACT(YEAR FROM to_timestamp(timestamp)) AS active_year
    FROM ml_ratings
),
retention AS (
    SELECT 
        f.first_year AS cohort_year,
        a.active_year - f.first_year AS year_offset,
        COUNT(DISTINCT a.userId) AS active_users
    FROM user_first_year f
    JOIN user_active_years a ON f.userId = a.userId
    WHERE f.first_year >= 2010 AND f.first_year <= 2015
      AND a.active_year >= f.first_year
      AND a.active_year <= f.first_year + 3
    GROUP BY f.first_year, a.active_year - f.first_year
)
SELECT 
    cohort_year,
    MAX(CASE WHEN year_offset = 0 THEN active_users END) AS year_0,
    MAX(CASE WHEN year_offset = 1 THEN active_users END) AS year_1,
    MAX(CASE WHEN year_offset = 2 THEN active_users END) AS year_2,
    MAX(CASE WHEN year_offset = 3 THEN active_users END) AS year_3,
    ROUND(100.0 * MAX(CASE WHEN year_offset = 1 THEN active_users END) / MAX(CASE WHEN year_offset = 0 THEN active_users END), 1) AS retention_y1_pct
FROM retention
GROUP BY cohort_year
ORDER BY cohort_year
""").df()

**这个查询的价值**：
- 数据岗面试官最爱问的就是**留存率怎么算**,这就是标准答案
- 这个表叫做"**Cohort Analysis（队列分析）**",简历项目里有这个会让面试官眼前一亮
- 当你 Day 5 建 ADS 层时,这就是一张典型的 `ads_user_retention` 表的产出 SQL

## 6.2 漏斗分析

**业务问题**：
- 全平台总用户数？
- 打过 >=1 部电影评分的用户？
- 打过 >=10 部电影评分的用户？
- 打过 >=50 部电影评分的用户？
- 打过 >=100 部电影评分的用户？

这就是"参与度漏斗"——你**易多保实习时设计的"浏览→加购→下单"漏斗思想**,放到这里就是"打 1 分→打 10 分→打 50 分"。

In [ ]:
con.sql("""
WITH user_counts AS (
    SELECT 
        userId,
        COUNT(*) AS rating_count
    FROM ml_ratings
    GROUP BY userId
)
SELECT 
    '总用户数' AS funnel_stage, COUNT(*) AS users, 1 AS step_order
    FROM user_counts
UNION ALL
SELECT '打过 >=10 部' AS funnel_stage, COUNT(*), 2 AS step_order
    FROM user_counts WHERE rating_count >= 10
UNION ALL
SELECT '打过 >=50 部' AS funnel_stage, COUNT(*), 3 AS step_order
    FROM user_counts WHERE rating_count >= 50
UNION ALL
SELECT '打过 >=100 部' AS funnel_stage, COUNT(*), 4 AS step_order
    FROM user_counts WHERE rating_count >= 100
UNION ALL
SELECT '打过 >=500 部' AS funnel_stage, COUNT(*), 5 AS step_order
    FROM user_counts WHERE rating_count >= 500
UNION ALL
SELECT '打过 >=1000 部' AS funnel_stage, COUNT(*), 6 AS step_order
    FROM user_counts WHERE rating_count >= 1000
ORDER BY step_order
""").df()

## 6.3 综合大题 Q13

### Q13：连续 N 天活跃用户

**业务问题**：找出所有"连续 3 天打过分"的用户。

**这是非常经典的面试题**,叫"连续登录题",标准答案模式：
1. 对每个用户的活跃日期排序,加 ROW_NUMBER
2. 用 `date - row_number天` 做一个魔术列,**连续的日期会变成同一个值**
3. GROUP BY userId + 魔术列,COUNT() 看是否 >= 3

**这是 SQL 经典魔术之一,搞懂这一题你会理解"用窗口函数 + 数学差值识别连续性"的精髓**

In [ ]:
# Q13 自己想想能不能写
con.sql("""

""").df()

In [ ]:
# Q13 参考答案 —— 经典 "连续登录" SQL
con.sql("""
WITH user_dates AS (
    SELECT DISTINCT
        userId,
        CAST(to_timestamp(timestamp) AS DATE) AS active_date
    FROM ml_ratings
    WHERE userId BETWEEN 1 AND 1000  -- 限定快速测试
),
ranked AS (
    SELECT 
        userId,
        active_date,
        ROW_NUMBER() OVER (PARTITION BY userId ORDER BY active_date) AS rn,
        -- 魔术：日期 - 排名,连续的日期减去递增的排名会得到相同的值
        active_date - INTERVAL (ROW_NUMBER() OVER (PARTITION BY userId ORDER BY active_date)) DAY AS magic_group
    FROM user_dates
),
streaks AS (
    SELECT 
        userId,
        magic_group,
        COUNT(*) AS streak_length,
        MIN(active_date) AS streak_start,
        MAX(active_date) AS streak_end
    FROM ranked
    GROUP BY userId, magic_group
    HAVING COUNT(*) >= 3  -- 至少连续 3 天
)
SELECT *
FROM streaks
ORDER BY streak_length DESC
LIMIT 20
""").df()

**消化一下这道题**：
- 这是 SQL 进阶的"小聪明"题之一
- 不用循环、不用过程式语言,纯 SQL 解决"连续"问题
- 这种模式叫 **"Gaps and Islands"**（缝隙与孤岛）,工作和面试中反复出现

---

# 🎉 Day 3 完成自检

扪心自问,以下问题你能在心里清晰回答吗？

1. WHERE 和 HAVING 的区别？
2. CTE 比子查询好在哪里？
3. ROW_NUMBER / RANK / DENSE_RANK 的区别？
4. 怎么算"每个分组的前 3 名"（TopN per group）？
5. 怎么算"最近 7 天的移动平均"？
6. 怎么算"用户留存率"的基本逻辑？
7. 怎么找"连续 N 天活跃的用户"？

如果 7 个里能答上 5 个,**Day 3 满分通过**。剩下 2 个回去翻一下对应的题就够了。

明天 Day 4 进 Hive,你写的这些 SQL **90% 能直接复用**——区别只是 Hive 跑得比 DuckDB 慢得多（但能处理 100 倍大的数据）。

**休息一晚,明天进入数仓世界。**